# Fabric / OneLake: write Power BI-ready predictions

Score the synthetic dataset and write a flat, typed predictions table. This
notebook uses the **local OneLake fallback** so it runs fully offline; set a
real Fabric workspace/lakehouse via env vars and the `fabric` extra to write to
OneLake. **All data is synthetic.**

In [ ]:
from revenue_prediction.config.loader import load_settings
from revenue_prediction.pipelines.local_pipeline import run_local_pipeline

settings = load_settings('dev')
output = run_local_pipeline(settings, output_dir='outputs')
print('Champion:', output.selection.champion)

In [ ]:
from revenue_prediction.core.data.synthetic import generate_synthetic_dataset
from revenue_prediction.core.inference.predict import batch_predict

df = generate_synthetic_dataset(settings.data)
predictions = batch_predict(output.champion_bundle, df, cutoff_day=settings.data.demo_cutoff_day)
predictions.head()

In [ ]:
from revenue_prediction.integrations.fabric.onelake import write_predictions_to_onelake

# local_root makes this offline-safe; drop it and configure RPA_FABRIC__* for real OneLake.
path = write_predictions_to_onelake(predictions, settings.fabric, local_root='outputs/onelake')
print('Wrote predictions to', path)

In Fabric, build a **DirectLake** semantic model over the predictions table and
a Power BI report. See `docs/fabric/integration.md`.